In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load Data
df = pd.read_csv("/kaggle/input/datasets/albertgomez601/aqi-2015-2024/city_hour.csv")
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values(['City', 'Datetime'])

# 2. Identify ALL numeric features (The "Train Everything" logic)
# We exclude non-numeric metadata
exclude = ['City', 'Datetime']
ALL_FEATURES = [col for col in df.columns if col not in exclude and pd.api.types.is_numeric_dtype(df[col])]
SEQ_LEN = 24 

# 3. Handle Missing Values per city
clean_chunks = []
for city in df['City'].unique():
    c_df = df[df['City'] == city].copy()
    c_df[ALL_FEATURES] = c_df[c_df['City'] == city][ALL_FEATURES].interpolate(method='linear').ffill().bfill()
    clean_chunks.append(c_df)
df = pd.concat(clean_chunks)

# 4. Scaling & Saving Metadata
scaler = MinMaxScaler()
df[ALL_FEATURES] = scaler.fit_transform(df[ALL_FEATURES])

# We save the feature names so the UI knows what to display
metadata = {
    'features': ALL_FEATURES,
    'scaler': scaler
}
with open("model_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

# 5. Sequence Creation
X, y = [], []
for city in df['City'].unique():
    city_vals = df[df['City'] == city][ALL_FEATURES].values
    if len(city_vals) > SEQ_LEN:
        for i in range(len(city_vals) - SEQ_LEN):
            X.append(city_vals[i : i + SEQ_LEN])
            y.append(city_vals[i + SEQ_LEN])

X, y = np.array(X), np.array(y)

# 6. Build Multi-Output Model
model = Sequential([
    Input(shape=(SEQ_LEN, len(ALL_FEATURES))),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dense(64, activation='relu'),
    Dense(len(ALL_FEATURES)) # Output layer matches total feature count
])

model.compile(optimizer='adam', loss='mse')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print(f"Training on {len(ALL_FEATURES)} features across all cities...")
model.fit(X, y, epochs=30, batch_size=64, validation_split=0.2, callbacks=[early_stop])

model.save("aqi_lstm_model.h5")
print("Done. Model and Metadata saved.")


2026-04-02 15:30:06.742422: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775143806.964153      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775143807.023327      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775143807.510976      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775143807.511026      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775143807.511029      24 computation_placer.cc:177] computation placer alr

Training on 13 features across all cities...
Epoch 1/30


I0000 00:00:1775143838.261372      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


5477/5477 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step - loss: 0.0855 - val_loss: 0.0836
Epoch 2/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 46s 8ms/step - loss: 0.0835 - val_loss: 0.0837
Epoch 3/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 45s 8ms/step - loss: 0.0833 - val_loss: 0.0836
Epoch 4/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 45s 8ms/step - loss: 0.0834 - val_loss: 0.0835
Epoch 5/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.0834 - val_loss: 0.0835
Epoch 6/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 45s 8ms/step - loss: 0.0834 - val_loss: 0.0835
Epoch 7/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.0833 - val_loss: 0.0835
Epoch 8/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 45s 8ms/step - loss: 0.0834 - val_loss: 0.0835
Epoch 9/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.0833 - val_loss: 0.0835
Epoch 10/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 44s 8ms/step - loss: 0.0833 - val_loss: 0.0835
Epoch 11/30
5477/5477 ━━━━━━━━━━━━━━━━━━━━ 43s 8ms/step - loss: 0.0833 - val_loss: 0.0835
Epoch 12/30
5477/5477 ━━━━━━━━

Done. Model and Metadata saved.
